# Codigo lambda 1: ingesta de API diaria
Esta lambda se lanza a diario gracias a un desencadenador de eventBridge

![Foto](img/1.png)

In [ ]:
import json
import boto3
import urllib3
import pandas as pd
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta

def lambda_handler(event, context):
    # Configuración
    s3 = boto3.client('s3')
    http = urllib3.PoolManager()
    
    bucket_name = 'bronce-hugoblancoalonso' # Sustituye por el tuyo
    
    # Parámetros de la API (Ejemplo: Balance eléctrico de 2024)
    url = "https://apidatos.ree.es/es/datos/balance/balance-electrico"

    hoy = date.today()
    params = {
        'start_date': f'{hoy}T00:00',
        'end_date': f'{hoy}T23:59',
        'time_trunc': 'day'
    }
    
    # Construcción de la URL con parámetros
    query_string = f"?start_date={params['start_date']}&end_date={params['end_date']}&time_trunc={params['time_trunc']}"
    full_url = url + query_string

    try:
        # Petición a la API
        response = http.request('GET', full_url)
        data = json.loads(response.data.decode('utf-8'))
        
        # Nombre del archivo con timestamp para evitar sobreescrituras
        file_name = f"bronce/esios/diario/balance_electrico_{datetime.now().strftime('%Y%m%d')}.json"
        
        # Guardar en S3
        s3.put_object(
            Bucket=bucket_name,
            Key=file_name,
            Body=json.dumps(data),
            ContentType='application/json'
        )
        
        return {
            'statusCode': 200,
            'body': json.dumps(f'Archivo guardado exitosamente: {file_name}')
        }
        
    except Exception as e:
        print(e)
        return {
            'statusCode': 500,
            'body': json.dumps('Error en la descarga o guardado de datos')
        }


# S3
Estos datos se envian y se almacenan en un bucket de s3

![foto](img/2.png)

# Codigo lambda 2: creacion json historico
Al almacenar los datos en el s3 esto hace una reaccion en cadena donde se ejecuta esta segunda lambda que crea un json historico con todos los datos del json diario

![foto](img/3.png)

In [ ]:
import json
import boto3
from datetime import datetime

def lambda_handler(event, context):
    s3 = boto3.client('s3')
    
    # --- Configuración ---
    bucket_name = 'bronce-hugoblancoalonso'
    prefix_diario = 'bronce/esios/diario/'
    key_historico = f'bronce/esios/historico/balance_electrico_historico.json' # Ajusta esta ruta
    
    try:
        # 1. Obtener el último archivo diario subido
        response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix_diario)
        if 'Contents' not in response:
            return {'statusCode': 404, 'body': 'No se encontraron archivos diarios.'}
        
        # Ordenar por fecha de última modificación para obtener el más reciente
        last_file = max(response['Contents'], key=lambda x: x['LastModified'])
        key_diario = last_file['Key']
        
        print(f"Procesando archivo diario: {key_diario}")

        # 2. Leer archivo diario y archivo histórico
        obj_diario = s3.get_object(Bucket=bucket_name, Key=key_diario)
        data_diaria = json.loads(obj_diario['Body'].read().decode('utf-8'))
        
        obj_historico = s3.get_object(Bucket=bucket_name, Key=key_historico)
        data_historica = json.loads(obj_historico['Body'].read().decode('utf-8'))

        # 3. Fusionar datos (Estructura REE: data -> included -> attributes -> values)
        # Mapeamos los datos diarios por el ID de la tecnología para facilitar la inserción
        nuevos_valores_por_id = {}
        for item in data_diaria.get('included', []):
            tech_id = item.get('id')
            values = item.get('attributes', {}).get('content', [{}])[0].get('attributes', {}).get('values', [])
            # O si el JSON diario es simplificado, ajusta esta ruta de acceso
            nuevos_valores_por_id[tech_id] = values

        # Insertar en el histórico
        for item_hist in data_historica.get('included', []):
            tech_id_hist = item_hist.get('id')
            if tech_id_hist in nuevos_valores_por_id:
                # Accedemos a la lista de valores del histórico
                # Según tu archivo, la ruta es: included -> attributes -> content -> attributes -> values
                # Nota: Tu JSON tiene una estructura anidada particular
                content_list = item_hist.get('attributes', {}).get('content', [])
                for content in content_list:
                    hist_values = content.get('attributes', {}).get('values', [])
                    # Añadimos los nuevos valores del día
                    hist_values.extend(nuevos_valores_por_id[tech_id_hist])
                    # (Opcional) Podrías añadir lógica aquí para evitar duplicados por fecha

        # 4. Guardar histórico actualizado
        s3.put_object(
            Bucket=bucket_name,
            Key=key_historico,
            Body=json.dumps(data_historica),
            ContentType='application/json'
        )

        return {
            'statusCode': 200,
            'body': json.dumps(f'Histórico actualizado con éxito usando {key_diario}')
        }

    except Exception as e:
        print(f"Error: {str(e)}")
        return {
            'statusCode': 500,
            'body': json.dumps(f'Error procesando los datos: {str(e)}')
        }